# IMPROVED DATA GENERATION - Step 1: Setup
Creates realistic synthetic healthcare data with distributions that prevent single algorithm solutions

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


# Step 2: Define Realistic Distribution Parameters

In [17]:
NUM_RECORDS = 7000

# Bimodal age distribution - realistic population
AGE_DIST = {
    'mean_young': 28,
    'std_young': 5,
    'mean_old': 55,
    'std_old': 8,
    'ratio': 0.6,
    'min_age': 18,
    'max_age': 75
}

REGIONS = ['North', 'South', 'East', 'West']
REGION_WEIGHTS = [0.35, 0.28, 0.22, 0.15]  # Unbalanced

ACTIVITIES = ['Walking', 'Gym', 'Running', 'Cycling', 'Sports']

print("✅ Distribution parameters defined")

✅ Distribution parameters defined


# Step 3: Generate Quasi-Identifiers (Age, Region, Activity)

In [18]:
def generate_age():
    """Generate bimodal age distribution with clustering"""
    young_count = int(NUM_RECORDS * AGE_DIST['ratio'])
    old_count = NUM_RECORDS - young_count
    
    young_ages = np.random.normal(
        AGE_DIST['mean_young'], 
        AGE_DIST['std_young'], 
        young_count
    )
    
    old_ages = np.random.normal(
        AGE_DIST['mean_old'], 
        AGE_DIST['std_old'], 
        old_count
    )
    
    ages = np.concatenate([young_ages, old_ages])
    ages = np.clip(ages, AGE_DIST['min_age'], AGE_DIST['max_age'])
    ages = np.random.permutation(ages.astype(int))
    
    return ages

ages = generate_age()
print(f"✅ Age generation complete")
print(f"   Age range: {ages.min()} - {ages.max()}")
print(f"   Mean age: {ages.mean():.1f}")

✅ Age generation complete
   Age range: 18 - 75
   Mean age: 38.2


# Step 4: Generate Regions and Activity (with correlation)

In [19]:
# Generate regions with unbalanced distribution
regions = np.random.choice(REGIONS, size=NUM_RECORDS, p=REGION_WEIGHTS)

# Generate activity correlated with age
# Older people tend to walk, younger people gym/running
activities = []
for age in ages:
    if age < 30:
        # Young people: prefer gym, running, sports
        activity = np.random.choice(
            ACTIVITIES, 
            p=[0.05, 0.35, 0.30, 0.20, 0.10]
        )
    elif age < 50:
        activity = np.random.choice(
            ACTIVITIES, 
            p=[0.10, 0.30, 0.20, 0.25, 0.15]
        )
    else:
        # Older people: prefer walking, less running
        activity = np.random.choice(
            ACTIVITIES, 
            p=[0.40, 0.25, 0.05, 0.20, 0.10]
        )
    activities.append(activity)

activities = np.array(activities)

print(f"✅ Region & Activity generation complete")
print(f"   Activity distribution:")
print(pd.Series(activities).value_counts())

✅ Region & Activity generation complete
   Activity distribution:
Gym        2120
Cycling    1587
Running    1285
Walking    1171
Sports      837
Name: count, dtype: int64


# Step 5: Generate Sensitive Attributes with Patterns

In [20]:
# Generate health metrics with realistic correlations
steps = []
for i, activity in enumerate(activities):
    age = ages[i]
    
    # Activity baseline
    if activity == 'Walking':
        base_steps = 8000
        std_steps = 2000
    elif activity == 'Gym':
        base_steps = 10000
        std_steps = 2500
    elif activity == 'Running':
        base_steps = 15000
        std_steps = 3000
    elif activity == 'Cycling':
        base_steps = 12000
        std_steps = 2500
    else:  # Sports
        base_steps = 13000
        std_steps = 2500
    
    # Age factor: older people do fewer steps
    age_factor = 1 - (age - 18) / 150
    age_factor = np.clip(age_factor, 0.5, 1.5)
    
    step_count = int(np.random.normal(base_steps * age_factor, std_steps))
    steps.append(max(1000, step_count))

steps = np.array(steps)

# Calories burned (correlated with steps and activity intensity)
calories = []
for step, activity, age in zip(steps, activities, ages):
    # Base calorie from steps (rough estimate: 0.04-0.05 per step)
    base_cal = int(step * (0.04 + np.random.normal(0, 0.005)))
    
    # Activity multiplier
    activity_mult = {
        'Walking': 0.9,
        'Gym': 1.1,
        'Running': 1.3,
        'Cycling': 1.2,
        'Sports': 1.25
    }
    
    cal = int(base_cal * activity_mult[activity] * (1 + np.random.normal(0, 0.05)))
    calories.append(max(100, cal))

calories = np.array(calories)

print(f"✅ Sensitive attributes generated")
print(f"   Steps range: {steps.min()} - {steps.max()}")
print(f"   Calories range: {calories.min()} - {calories.max()}")

✅ Sensitive attributes generated
   Steps range: 1000 - 22701
   Calories range: 100 - 1444


# Step 6: Generate Heart Rate & Sleep with Outliers

In [21]:
# Heart rate generation with age correlation and some anomalies
heart_rates = []
for i, (age, activity) in enumerate(zip(ages, activities)):
    # Base resting heart rate increases with age
    base_hr = 60 + (age - 18) * 0.1
    
    # Activity adds variation
    activity_effect = {
        'Walking': 10,
        'Gym': 20,
        'Running': 40,
        'Cycling': 25,
        'Sports': 30
    }
    
    # Random outliers (5% chance of anomaly - could indicate health issues)
    if np.random.random() < 0.05:
        # Outliers: unusual heart rates
        hr = np.random.choice([45, 50, 110, 120, 130])
    else:
        activity_hr = base_hr + activity_effect[activity]
        hr = int(np.random.normal(activity_hr, 8))
        hr = np.clip(hr, 40, 150)
    
    heart_rates.append(hr)

heart_rates = np.array(heart_rates)

# Sleep hours (inverse correlation with exercise intensity)
sleep_hours = []
for activity in activities:
    activity_fatigue = {
        'Walking': -0.5,
        'Gym': 0.0,
        'Running': 0.3,
        'Cycling': 0.2,
        'Sports': 0.2
    }
    
    # Base sleep + activity effect + noise
    sleep = 7 + activity_fatigue[activity] + np.random.normal(0, 1)
    sleep = np.clip(sleep, 3, 12)
    sleep_hours.append(round(sleep, 2))

sleep_hours = np.array(sleep_hours)

print(f"✅ Heart rate & sleep generated")
print(f"   Heart rate range: {heart_rates.min()} - {heart_rates.max()}")
print(f"   Sleep range: {sleep_hours.min()} - {sleep_hours.max():.1f} hours")
print(f"   Outliers in heart rate: {sum(heart_rates > 110) + sum(heart_rates < 50)}")

✅ Heart rate & sleep generated
   Heart rate range: 44 - 130
   Sleep range: 3.4 - 10.5 hours
   Outliers in heart rate: 358


# Step 7: Generate Weight (highly personal, minimal correlation)

In [22]:
# Weight with slight age correlation but mostly random
# This creates natural variation within QI groups
weights = []
for age in ages:
    # Slight increase with age
    base_weight = 70 + (age - 18) * 0.08
    
    # High variance in weight
    weight = np.random.normal(base_weight, 15)
    weight = np.clip(weight, 45, 150)
    
    weights.append(round(weight, 2))

weights = np.array(weights)

print(f"✅ Weight generated")
print(f"   Weight range: {weights.min()} - {weights.max()} kg")

✅ Weight generated
   Weight range: 45.0 - 127.98 kg


# Step 8: Combine into DataFrame

In [23]:
df = pd.DataFrame({
    'age': ages,
    'region': regions,
    'activity': activities,
    'steps': steps,
    'calories': calories,
    'heart_rate': heart_rates,
    'weight': weights,
    'sleep_hours': sleep_hours
})

print(f"✅ DataFrame created")
print(f"\nDataset shape: {df.shape}")
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nData types:")
print(df.dtypes)
print(f"\nBasic statistics:")
print(df.describe())

✅ DataFrame created

Dataset shape: (7000, 8)

First 5 rows:
   age region activity  steps  calories  heart_rate  weight  sleep_hours
0   22   East      Gym  10170       433          85   78.11         5.81
1   20  South  Running  11890       545         105   83.26         5.94
2   50  North      Gym   6461       270          62   64.61         7.19
3   29  North      Gym  12858       686          77   63.33         8.29
4   31  North  Walking  11024       486          68   86.21         8.96

Data types:
age              int64
region             str
activity           str
steps            int64
calories         int64
heart_rate       int64
weight         float64
sleep_hours    float64
dtype: object

Basic statistics:
               age         steps     calories   heart_rate       weight  \
count  7000.000000   7000.000000  7000.000000  7000.000000  7000.000000   
mean     38.244714   9935.002857   465.575143    86.235000    71.828874   
std      14.466963   3594.916670   212.195423 

# Step 9: Save the Dataset

In [25]:
# Save to CSV
import os
os.makedirs('../data', exist_ok=True)

df.to_csv('../data/raw_fitness_data.csv', index=False)
print(f"✅ Dataset saved to '../data/raw_fitness_data.csv'")
print(f"\nDataset characteristics that prevent single-algo solutions:")
print(f"  1. Bimodal age distribution (creates sparse groups)")
print(f"  2. Unbalanced region distribution (some regions sparse)")
print(f"  3. Correlated quasi-identifiers (can't just generalize one)")
print(f"  4. High variance in sensitive attributes (similar values across groups)")
print(f"  5. Outliers in health metrics (5% anomalies in heart rate)")

✅ Dataset saved to '../data/raw_fitness_data.csv'

Dataset characteristics that prevent single-algo solutions:
  1. Bimodal age distribution (creates sparse groups)
  2. Unbalanced region distribution (some regions sparse)
  3. Correlated quasi-identifiers (can't just generalize one)
  4. High variance in sensitive attributes (similar values across groups)
  5. Outliers in health metrics (5% anomalies in heart rate)


# Step 10: Analyze the Problem Structure

In [27]:
# Check k-anonymity violations
quasi_ids = ['age', 'region', 'activity']
k = 5

group_sizes = df.groupby(quasi_ids).size()
violations = (group_sizes < k).sum()
total_groups = len(group_sizes)

print(f"K-Anonymity Analysis (k={k}):")
print(f"  Groups violating k-anonymity: {violations}/{total_groups}")
print(f"  Violation rate: {100*violations/total_groups:.1f}%")

# Check l-diversity (assuming 'heart_rate' as sensitive)
print(f"\nL-Diversity Analysis (heart_rate as sensitive):")
for qi_cols in [quasi_ids]:
    diversity = df.groupby(qi_cols)['heart_rate'].nunique()
    low_diversity = (diversity < 3).sum()
    print(f"  Groups with < 3 unique values: {low_diversity}/{len(diversity)}")

# Check t-closeness
print(f"\nT-Closeness Analysis (calories as sensitive):")
for qi_cols in [quasi_ids]:
    diversity = df.groupby(qi_cols)['calories'].nunique()
    print(f"  Average unique values per group: {diversity.mean():.1f}")

K-Anonymity Analysis (k=5):
  Groups violating k-anonymity: 499/1001
  Violation rate: 49.9%

L-Diversity Analysis (heart_rate as sensitive):
  Groups with < 3 unique values: 283/1001

T-Closeness Analysis (calories as sensitive):
  Average unique values per group: 6.9
